# CPSA 05 — QAT probe and Brevitas fallback

The baseline is FP32 → Vitis AI PTQ. This notebook probes `pytorch_nndct`
and, when unavailable, wraps the Hard-Swish model with Brevitas fixed-point
fake quantizers. Brevitas is not used for export; the adapted weights go
through the same Vitis PTQ flow.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "a06184367d62061955867d6199b949383e659763"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "brevitas==0.13.4"],
    check=True,
)

In [ ]:
from pathlib import Path

import torch

from cpsa_ml.datasets.staging import find_inputs
from cpsa_ml.models import load_yolo
from cpsa_ml.qat import brevitas_available, quantize_model, try_nndct

SCHEMA = "halfbust62"
WEIGHTS = next(
    iter(find_inputs(Path("/kaggle/input"), "best.pt")), Path("/kaggle/input/missing-best.pt")
)
probe = try_nndct()
print("pytorch_nndct:", probe)

if probe:
    print("Use pytorch_nndct QAT; record the Task 8 install-spike result.")
elif brevitas_available():
    if WEIGHTS.is_file():
        from ultralytics import YOLO

        yolo = YOLO(str(WEIGHTS), verbose=False)
    else:
        yolo = load_yolo(SCHEMA, variant="n", activation="hswish")
    qmodel = quantize_model(yolo.model.eval())
    with torch.no_grad():
        qmodel(torch.randn(1, 3, 480, 640))
    torch.save(qmodel.state_dict(), "/kaggle/working/qat_weights.pt")
    print("Brevitas layer-wise QAT wrapper ready: /kaggle/working/qat_weights.pt")
else:
    print("Brevitas unavailable; use the FP32 -> Vitis PTQ baseline.")

In [ ]:
# Publish /kaggle/working/qat_weights.pt (or the nndct checkpoint) as a dataset
# version for the Task 9 Vitis flow.
print("QAT artifacts are under /kaggle/working")